In [1]:
import numpy as np

In [ ]:
n = 101
a = 2
b = 5
c = 7

## create nxn identity matrix
I = np.eye(n)
K_diag = np.diag(np.full(n, a), k=0) + np.diag(np.full(n-1, b), k=1) + np.diag(np.full(n-2, c), k=2)
K = np.vstack((I[0, :], K_diag[:-1, :]))
K[-1, :] = I[-1, :] # replace last row with identity matrix row
K

array([[1., 0., 0., ..., 0., 0., 0.],
       [2., 5., 7., ..., 0., 0., 0.],
       [0., 2., 5., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 5., 7., 0.],
       [0., 0., 0., ..., 2., 5., 7.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(101, 101))

In [92]:
K = np.eye(n)
row = np.concatenate((np.array([a, b, c]), np.zeros(n-3)))
for i in range(1, n-1):
    k_row = np.roll(row, i-1)
    K[i, :] = k_row

K

array([[1., 0., 0., ..., 0., 0., 0.],
       [2., 5., 7., ..., 0., 0., 0.],
       [0., 2., 5., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 5., 7., 0.],
       [0., 0., 0., ..., 2., 5., 7.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(101, 101))

***
:::{admonition} implicit update rule

Let us derive the **update rule** for this new scheme and **formulate the system of equations** for $n$ grid points, considering the boundary conditions. 

Starting with the above approximate version of the wave equation and substituting the second-order central-difference approximations for the second derivatives in time and space, we obtain
$$
\frac{{u}_j^{\alpha-1}-2{u}_j^\alpha+{u}_j^{\alpha+1}}{\Delta t^2}=\frac{c^2}{2}\left[\frac{{u}_{j-1}^{\alpha-1}-2{u}_j^{\alpha-1}+{u}_{j+1}^{\alpha-1}}{\Delta {x}^2}+\frac{{u}_{j-1}^{\alpha+1}-2{u}_j^{\alpha+1}+{u}_{j+1}^{\alpha+1}}{\Delta {x}^2}\right]+ O(\Delta x^2,\Delta t^2).
$$

Moving all terms at the next timestep $t^{\alpha+1}$ to the left-hand side and using the abbreviation $r = (c\frac{\Delta t}{\Delta x}) ^{2}$ leads to

$$
u_{j}^{\alpha + 1} (1 + r) - \frac{r}{2} (u_{j+1}^{\alpha+1} + u_{j-1}^{\alpha+1} )= 2 u_{j}^{\alpha} - u_{j}^{\alpha-1} + \frac{r}{2} \left(  u_{j+1}^{\alpha-1} - 2u_{j}^{\alpha-1}  + u_{j-1}^{\alpha-1}   \right).
$$

This is an *implicit* scheme because we cannot explicitly solve directly for the next time step at each node (the solution for the next time step at any one node depends on the solution at the next time step at all the other nodes). Therefore, we must solve for the displacements $u$ at the next time step for all nodes at once. This involves solving a linear system of equations ($\boldsymbol{K}\boldsymbol{U}_{new} =\boldsymbol{F}$) at each time step. If there are $n$ nodes, then we can assemble the system of equations into matrix form as

$$
	\left[\begin{array}{ccccccc}
		1 & 0 & 0 & 0 & 0 & 0 & 0 \\
		-\frac{r}{2}& (1+r) & -\frac{r}{2} & 0 & 0 & 0 & 0 \\
		0 & -\frac{r}{2} & (1+r) & -\frac{r}{2} & 0 & 0 & 0 \\
		0 & 0 & \cdot & \cdot & \cdot & 0 & 0 \\
		0 & 0 & 0 & -\frac{r}{2} & (1+r)& -\frac{r}{2} & 0 \\
		0 & 0 & 0 & 0 & -\frac{r}{2}& (1+r) & -\frac{r}{2} \\
		0 & 0 & 0 & 0 & 0 & 0 & 1
	\end{array}\right]
	\left[\begin{array}{c}
		{u}_0^{\alpha+1} \\
		{u}_1^{\alpha+1} \\
		{u}_2^{\alpha+1} \\
		\cdot \\
		\cdot \\
		{u}_{n-2}^{\alpha+1} \\
		{u}_{n-1}^{\alpha+1}
	\end{array}\right]
	=\boldsymbol{F}
$$

where the right-hand-side is given by

$$
	\boldsymbol{F}=
	\left[\begin{array}{c}
		\hat{u}({t}) \\
		2 u_{1}^{\alpha} - u_{1}^{\alpha-1} + \frac{r}{2} \left(  u_{2}^{\alpha-1} - 2u_{1}^{\alpha-1}  + u_{0}^{\alpha-1}   \right) \\ 
		\cdot \\
		2 u_{j}^{\alpha} - u_{j}^{\alpha-1} + \frac{r}{2} \left(  u_{j+1}^{\alpha-1} - 2u_{j}^{\alpha-1}  + u_{j-1}^{\alpha-1}   \right) \\
		\cdot \\
		2 u_{n-1}^{\alpha} - u_{n-1}^{\alpha-1} + \frac{r}{2} \left(  u_{n}^{\alpha-1} - 2u_{n-1}^{\alpha-1}  + u_{n-2}^{\alpha-1}   \right) \\
		0
	\end{array}\right].
$$

Note that we implemented the two Dirichlet (essential) boundary conditions at both ends of the bar by modifying the first and last row in the above system of equations, as discussed in class.

:::

# Structure Analysis of Bars and Trusses

In [13]:
# cosine of pi/4 as a floating point number rounded to 2dp
theta = np.cos(np.pi/2)
int(theta)
# show as floating point number rounded to 2 decimal places

0

In [17]:
nodal_connectivities = np.array([[0, 1], [1, 2], [0, 2]])
element_dof_connectivities = np.array([[node[0]*2, node[0]*2+1,node[1]*2, node[1]*2+1] for node in nodal_connectivities])
element_dof_connectivities

array([[0, 1, 2, 3],
       [2, 3, 4, 5],
       [0, 1, 4, 5]])

In [19]:
np.zeros((6,6))

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])